# H03 — Homework: the noise of a mirror on a spring

> **References**
>
> (Therrien 1992), (Papoulis and Pillai 2002), (Kay 1988), (Harris 1978)

## What you do here

**The system.** The mirror of H02 on its spring. The ground under it is still.

**The noise that drives it.**
- Gas molecules hit the mirror at random.
- Each kick is independent of the others.
- So the force $F$ is **white noise**: nothing links one sample to the next.

**What the mirror does.** It answers through $G_{F}(f)$, from the force to the position. Its motion is **colored noise**: it remembers its past and it rings near $f_{0}$. The noise of a real detector is made in this way.

**What we measure.** The motion in time, with the autocorrelation $r_{xx}[k]$ and in frequency, with the power spectral density $S_{xx}(f)$. Both from the one record we have.

**How you work in each task.**
1. Look at the steps of the task.
2. Write a few lines in the empty cell; the rest is given and commented.
3. Change one parameter under `# change these` and run again.
4. Answer the question, then open the solution.

The exercise on a circuit at the end is yours alone: no code is given.

> **The symbols**
>
> - **$m$**, **$k_{\mathrm{s}}$**, **$b$** — the mass of the mirror (kg), the spring constant (N/m) and the damping (kg/s), as in H02; the letter $k$ alone is the lag
> - **$\omega_{0}$**, **$Q$**, **$\tau_{0}$** — the frequency, the quality factor and the ringdown time of the mirror, as in H02
> - **$F[n]$**, **$x[n]$** — the force on the mirror and the position of the mirror, in arbitrary units (a.u.), sampled every $\Delta t$ seconds; $n$ counts the samples
> - **$G_{F}(f)$** — the transfer function from the force to the position, $X(f) = G_{F}(f)\,F(f)$, in m/N. It is the $G(f)$ of H02 divided by $k_{\mathrm{s}}$
> - **$r_{xx}[k]$** — the autocorrelation of $x$ at the lag $k$, in steps
> - **$S_{F}$**, **$S_{xx}(f)$** — the power spectral densities of the force and of the mirror: one sided, in a.u.$^{2}$ per hertz. They are $2\,\Delta t$ times the two-sided densities per cycle per sample of lecture 9

| Part | The question | Tasks |
|---|---|---|
| The process and its memory | What is a random process? How far does its memory reach? | 1 and 2 |
| Where the power is | How is the power spread in frequency? | 3 |
| The estimate from one record | How do we measure the spectrum from one record? How long must a segment be? | 4 |
| In class, on your own | Does the same noise appear in a circuit? | exercise B |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import signal
import sys, os

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)

from gwcourse_palette import P, apply_style
apply_style()

The workflow of this laboratory: the mirror kicked by the gas, its two descriptions and the estimate from one record.

![The mirror on its spring with the gas kicking it. The chain force F, system G_F(f), mirror x. The two descriptions of one record, the autocorrelation in time and the density in frequency. Below, the estimate: record, segments, window, squared transform, average.](figures/h03_workflow.png)

> **How to use this notebook**
>
> Run the cells in order. In every task you write the code: a few lines in the empty cell, then compare with the expected result and open the solution. Every task has the same parts: what we do, the functions and the cell with the parameters under `# change these` (both given), your code, a box **What to change**, a question and a solution. A function is defined in the task where it is first needed. A later task that uses it says so. The random numbers are seeded: a run gives the numbers of the solution. Change one parameter at a time and run the cells again, then set it back before the next one. Answer the question before you open the solution. Every cell runs in less than 20 seconds.

## The process and its memory

The force has no memory. The mirror has. We see first what a random process is and then how far its memory reaches.

### Task 1 · Many records and one record

**What we do.** We make $R$ records of the mirror from $R$ different forces. Then we compare two averages.

**The force.** A new Gaussian number $F[n]$ at every step, with rms $\sigma_{F}$.

**The mirror.** It answers through

$$
G_{F}(f) = \frac{1}{m\left(\omega_{0}^{2} - \omega^{2} + \mathrm{i}\,\omega\,\omega_{0}/Q\right)},
\qquad \omega = 2\pi f .
$$

The product $X(f) = G_{F}(f)\,F(f)$ gives the position $x[n]$, sampled every $\Delta t = 0.02$ s.

**The records.** We get $R$ **realizations** $x^{(i)}[n]$ of one process. At a fixed step $n$ the $R$ values are a random variable.

**The two averages.**
- The **expectation** $\mathcal{E}\{x[n]\}$ is the average over infinitely many records. The average across our $R$ records is an estimate of it.
- The **time average** is the average along one record.

**Why it matters.** A detector gives one record. So we need the two averages to agree: the process must be **ergodic**.

**Gaussian.** The force is Gaussian and the mirror is linear. A sum of Gaussian numbers is Gaussian. So the position of the mirror at a fixed $n$ is Gaussian too.

**The figures.**
- The first draws one force record and four records of the mirror with the average across the records.
- It draws also the histogram across the records at $n_{\text{fixed}}$, the step marked by the dotted line.
- The second compares the two averages.

> **The symbols**
>
> - **$\sigma_{F}$** — the rms of the force in one sample, in a.u.
> - **$\omega$** — the angular frequency, $\omega = 2\pi f$
> - **$R$** — the number of records; **$T$** — the length of one record in seconds, so $N = T/\Delta t$ samples
> - **$\mathcal{E}\{\cdot\}$** — the expectation: the average across infinitely many records at a fixed $n$; for the mirror it is 0

The steps of this task.

![Many forces go through the mirror and give many records. The two averages, across the records and along one record.](figures/h03_task1_route.png)

In [ ]:
def oscillator_parameters(m, k_spring, b):
    """Return the natural frequency, the quality factor and the decay time of the mirror (as in H02).

    Parameters
    ----------
    m        : float, mass of the mirror (kg)
    k_spring : float, spring constant (N/m)
    b        : float, damping coefficient (kg/s)

    Returns
    -------
    (f0, q, tau0) : f0 natural frequency (Hz), q quality factor (no unit), tau0 ring-down time (s)
    """
    # natural frequency, quality factor and decay time of the amplitude
    f0 = np.sqrt(k_spring / m) / (2 * np.pi)
    q = np.sqrt(k_spring * m) / b
    tau0 = 2 * m / b
    params = (f0, q, tau0)
    return params


def transfer_function(f, m, k_spring, b):
    """Return the transfer function G_F = X / F from the force to the position of the mirror.

    It is the G(f) of H02 divided by k_s.

    Parameters
    ----------
    f        : float or array, frequency (Hz)
    m        : float, mass (kg)
    k_spring : float, spring constant (N/m)
    b        : float, damping coefficient (kg/s)

    Returns
    -------
    G_F : complex, same shape as f, in m/N; |G_F| is the amplitude gain
    """
    # G_F = 1 / (m (w0^2 - w^2 + i w w0 / Q)), written with k_s = m w0^2 and b = m w0 / Q
    w = 2 * np.pi * f
    G_F = 1 / (k_spring - m * w ** 2 + 1j * b * w)
    return G_F


def kicked_mirror(R, N, dt, m, k_spring, b, sigma_F, seed=2026):
    """Return R records of a white force and of the mirror that it pushes.

    Uses transfer_function. The mirror is found in frequency, as in task 3 of H02: transform of the
    force, product with G_F, transform back. The ringing at the end of a record wraps round to its
    start: for a record much longer than tau0 this is negligible and every sample is stationary.

    Parameters
    ----------
    R, N     : number of records and of samples in each (no unit)
    dt       : float, time step (s)
    m, k_spring, b : mass (kg), spring constant (N/m), damping coefficient (kg/s)
    sigma_F  : float, rms of the force in one sample (a.u.)
    seed     : integer, seed of the random numbers

    Returns
    -------
    (force, x) : two arrays of shape (R, N), the force and the position of the mirror (a.u.); row i is the record i
    """
    # the force: a new Gaussian number at every step of every record
    rng = np.random.default_rng(seed)
    force = sigma_F * rng.normal(size=(R, N))
    # to frequency, times the transfer function, back to time
    f = np.fft.rfftfreq(N, dt)
    X = transfer_function(f, m, k_spring, b) * np.fft.rfft(force, axis=-1)
    x = np.fft.irfft(X, N, axis=-1)
    records = (force, x)
    return records

In [ ]:
# change these: the mirror, the force and the records
m = 1.0            # mass of the mirror (kg)
k_spring = 39.5    # spring constant (N/m): f0 = sqrt(k_s / m) / 2 pi, 1 Hz here
b = 0.63           # damping coefficient (kg/s): Q = sqrt(k_s m) / b, 10 here
sigma_F = 40.0     # rms of the force in one sample (a.u.)
dt = 0.02          # time step (s)
R = 400            # number of records
T = 120.0          # length of one record (s)
n_fixed = 1000     # the step at which we look across the records
n_shown = 4        # records drawn in the figure

# R records of the force and of the mirror
f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
N = int(T / dt)
force, x = kicked_mirror(R, N, dt, m, k_spring, b, sigma_F)
print(f"f0 = {f0:.3f} Hz   Q = {Q:.1f}   tau0 = {tau0:.2f} s   {R} records of {N} samples")
print(f"rms of the force {force.std():.2f} a.u.")

**Your code.**
- `x` is an array with $R$ rows and $N$ columns: row $i$ is the record $i$.
- Write two lines. `expectation` is the average across the records at every $n$: an array of $N$ values. `time_average` is the average along every record: an array of $R$ values.
- Use `x.mean(axis=...)`.
- Then print the rms of the mirror (`x.std()`) and the standard deviation of each average.

**What you should see.**
- Both averages are near 0, the expectation of the mirror.
- The rms of the mirror is about $0.80$ a.u.
- The average across the records scatters over $n$ by about the rms divided by $\sqrt{R}$, which is $0.040$.
- The time averages scatter by a smaller number, about $0.013$.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
# the average across the records at every step n, and the average along every record
expectation = x.mean(axis=0)
time_average = x.mean(axis=1)

# the rms of the mirror and the scatter of the two averages
sigma_x = x.std()
print(f"rms of the mirror {sigma_x:.4f} a.u.")
print(f"scatter of the average across the records, over n: {expectation.std():.4f}   sigma_x / sqrt(R) = {sigma_x / np.sqrt(R):.4f}")
print(f"scatter of the time averages, over the records:    {time_average.std():.4f}")
```
**In one line.** `x.mean(axis=0)` averages across the records and `x.mean(axis=1)` along each record. Both estimate 0 and both are close to it.

| quantity | value |
|---|---|
| rms of the mirror | $0.8045$ a.u. |
| scatter over $n$ of the average across the records | $0.0383$, close to $\sigma_{x}/\sqrt{R} = 0.0402$ |
| scatter of the time averages from one record to the next | $0.0128$ |

**Why.**
- `x.mean(axis=0)` averages over the rows: at every step $n$ it averages the $R$ records.
- `x.mean(axis=1)` averages over the columns: for every record it averages along $n$.

</details>


In [ ]:
# first figure: one force record, a few mirror records with the expectation, the histogram across the records
t = np.arange(N) * dt
shown = t < 30
sigma_x = x.std()
across = x[:, n_fixed]
colours = [P['BLUE'], P['SAGE'], P['AMBER'], P['PLUM']]
fig, axs = plt.subplot_mosaic([['g', 'h'], ['x', 'h']], figsize=(11, 6), gridspec_kw={'width_ratios': [3, 1]})
axs['g'].plot(t[shown], force[0][shown], color=P['GREY_MID'], lw=0.7)
axs['g'].set(ylabel='force $F$ (a.u.)', xticklabels=[], xlim=(0, 30))
for i in range(n_shown):
    axs['x'].plot(t[shown], x[i][shown], color=colours[i % 4], lw=0.9, label=f'record {i + 1}')
axs['x'].plot(t[shown], expectation[shown], color=P['BRICK'], lw=1.8, label='average across the records')
axs['x'].axvline(n_fixed * dt, color=P['GREY_MID'], lw=1.0, ls=':')
axs['x'].set(xlabel='time (s)', ylabel='mirror $x$ (a.u.)', xlim=(0, 30))
axs['x'].legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=n_shown + 1, frameon=False, fontsize=9)
v = np.linspace(-4 * sigma_x, 4 * sigma_x, 200)
axs['h'].hist(across, bins=25, density=True, color=P['AMBER_LIGHT'], edgecolor=P['AMBER'], orientation='horizontal')
axs['h'].plot(np.exp(-0.5 * (v / sigma_x) ** 2) / (sigma_x * np.sqrt(2 * np.pi)), v, color=P['BRICK'], lw=1.6)
axs['h'].set(xlabel=f'density at $n = {n_fixed}$', ylabel='$x$ (a.u.)')
axs['h'].yaxis.set_label_position('right')
axs['h'].yaxis.tick_right()
plt.tight_layout()
plt.show()

# the Gaussian check at the fixed step
print(f"at n = {n_fixed}: fraction of the {R} records within 1 sigma {np.mean(np.abs(across) < sigma_x):.2f}"
      f" and within 2 sigma {np.mean(np.abs(across) < 2 * sigma_x):.3f}   (Gaussian: 0.68 and 0.954)")

# second figure: the values of the two averages on the same axes
fig, ax = plt.subplots(figsize=(9, 3.4))
half = 4 * sigma_x / np.sqrt(R)
bins = np.linspace(-half, half, 41)
ax.hist(time_average, bins=bins, density=True, histtype='step', color=P['BLUE'], lw=1.8, label=f'time averages, {R} values')
ax.hist(expectation, bins=bins, density=True, histtype='step', color=P['BRICK'], lw=1.8, label=f'averages across the records, {N} values')
ax.axvline(0, color=P['GREY_MID'], lw=1.0, ls=':')
ax.set(xlabel='value of the average (a.u.)', ylabel='density')
ax.legend(loc='lower right', bbox_to_anchor=(1.0, 1.0), ncol=2, frameon=False)
plt.show()

**What to change.** On the line `# change these`, one parameter at a time. Run the cells of the task again each time.
1. Nothing first. In the first figure compare the gray record of the force with the colored records of the mirror. Then read the printed lines.
2. `R`: 100 and 1600. Which curve of the second figure becomes narrower?
3. `T`: 30 and 480 s. Which curve becomes narrower now?
4. `b` (try 0.21 and 2.1 kg/s): how do the rms of the mirror and the two scatters change?

**Question.** Which average becomes narrower when $R$ grows and which when $T$ grows? When can one record give us the expectation?

<details>
<summary>Solution</summary>

**In one line.** A larger $R$ narrows the average across the records and a longer $T$ narrows the time average. One record gives the expectation when it is much longer than the memory $\tau_{0}$.

**The first run** ($R = 400$ records of 120 s).
- The mirror has an rms of $0.804$ a.u. and the force has $40.0$.
- The gray force jumps from one sample to the next. The mirror wanders at about 1 Hz: neighboring samples are linked.
- At $n = 1000$ the histogram across the records is Gaussian: $0.70$ of the records are within $1\sigma$ and $0.945$ within $2\sigma$, against $0.68$ and $0.954$.
- At $n = 10$ and $n = 5000$ the fractions are $0.67$ and $0.69$ within $1\sigma$: the process does not change with $n$.

**The scatters.**

| change | rms of the mirror | across the records | time averages |
|---|---|---|---|
| none ($R = 400$, $T = 120$ s) | $0.804$ | $0.038$ | $0.0128$ |
| $R = 1600$ | | $0.018$ | $0.0133$ |
| $R = 100$ | | $0.088$ | |
| $T = 30$ s | | $0.034$ | $0.026$ |
| $T = 480$ s | | $0.038$ | $0.0066$ |
| $b = 0.21$ kg/s | $1.40$ | $0.067$ | $0.0128$ |
| $b = 2.1$ kg/s | $0.440$ | $0.021$ | $0.0128$ |

- A larger $R$ does not move the time averages.
- A longer record does: the scatter falls as $1/\sqrt{T}$.
- The average across the records does not follow $T$.

**Why.** The time average is dominated by the slowest motions. There the mirror follows the force: $G_{F} = 1/k_{\mathrm{s}}$ whatever $b$ is.

**When one record is enough.**
- The expectation needs many records and a detector has one.
- Here the process does not change with $n$ (stationary) and each record runs through all its values (ergodic).
- The time average of one record then tends to the expectation as $1/\sqrt{T}$.
- A record is long enough when it is long compared with the memory of the process: here $\tau_{0} = 3.17$ s and the record is 38 times longer.
- A short record is not enough: at $T = 30$ s the time average scatters by $0.026$.

</details>


### Task 2 · The memory

**Uses** `oscillator_parameters` and `kicked_mirror` from task 1.

**What we do.** Take one record of the mirror. Plot each sample $x[n+k]$ against the sample $x[n]$ that is $k$ steps before.
- If the cloud is a line, $x[n]$ tells us what comes $k$ steps later: the samples are strongly correlated.
- If the cloud is round they are not linked.

**The autocorrelation.** It collects this in one number for every lag:

$$
r_{xx}[k] = \mathcal{E}\{x[n]\,x[n+k]\} .
$$

**How we estimate it.** The mean of the mirror is zero and the process is ergodic. So $r_{xx}[k]$ is the time average of $x[n]\,x[n+k]$ along one long record:

$$
r_{xx}[k] \approx \frac{1}{N-k}\sum_{n=0}^{N-1-k} x[n]\,x[n+k] .
$$

At $k = 0$ it is the variance.

**What we expect.**
- The force has no memory: each kick is new.
- The mirror has memory. A kick starts a ringing at $f_{0}$ that decays in the time $\tau_{0} = 2m/b$. The next kicks add to the ringing that is still there.
- So the autocorrelation of the mirror rings at $f_{0}$ inside the envelope $\mathrm{e}^{-k\Delta t/\tau_{0}}$ of H02.

**The figure.** Above, the scatter for three lags. Below, $r_{xx}[k]/r_{xx}[0]$ for the force and for the mirror with three values of $b$. The dashed lines are $\pm\,\mathrm{e}^{-k\Delta t/\tau_{0}}$.

> **The symbols**
>
> - **$k$** — the lag, in steps; the time distance is $k\,\Delta t$
> - **$r_{xx}[k]$** — the autocorrelation of one record $x$, in a.u.$^{2}$; $r_{xx}[0]$ is its variance

The steps of this task.

![One record, a copy shifted by k steps, the average of their product, the autocorrelation for every k.](figures/h03_task2_route.png)

In [ ]:
def autocorrelation(x, k_max):
    """Return the autocorrelation r_xx[k], k = 0 ... k_max, of one record: the time average of x[n] x[n+k].

    This is the loop of your cell done for all the lags at once with transforms, so that it is fast.

    Parameters
    ----------
    x     : array, one record of a process with zero mean (a.u.)
    k_max : int, the largest lag (steps)

    Returns
    -------
    r : array of k_max + 1 values, the autocorrelation (a.u.^2); r[0] is the variance of the record
    """
    # the sums of x[n] x[n+k] for all the lags at once: transform, square, transform back (the padding avoids wrap-around)
    N = len(x)
    X = np.fft.rfft(x, 2 * N)
    sums = np.fft.irfft(np.abs(X) ** 2)[:k_max + 1]
    # each lag has N - k pairs
    r = sums / (N - np.arange(k_max + 1))
    return r

In [ ]:
# change these: the mirror, with three dampings, and the lags of the scatter plots
m = 1.0                    # mass of the mirror (kg)
k_spring = 39.5            # spring constant (N/m): f0 = 1 Hz
b = 0.63                   # damping coefficient of the mirror you work with (kg/s): Q = 10
b_others = (1.26, 0.31)    # two more dampings for the figure (kg/s): Q = 5 and 20
lags_s = (0.04, 0.5, 10.0) # lags of the three scatter plots (s), multiples of dt
sigma_F = 40.0             # rms of the force in one sample (a.u.)
dt = 0.02                  # time step (s)
T = 1000.0                 # length of the record (s)
lag_max_s = 15.0           # largest lag drawn in the autocorrelation (s)

# one long record of the force and of the mirror for every damping; the same force pushes all of them
N = int(T / dt)
k_max = int(lag_max_s / dt)
b_values = (b_others[0], b, b_others[1])
records = [kicked_mirror(1, N, dt, m, k_spring, bb, sigma_F) for bb in b_values]
force = records[1][0][0]
x = records[1][1][0]
print(f"one record of {N} samples; the lags go up to k_max = {k_max} steps = {lag_max_s:.0f} s")

**Your code.**
- `force` and `x` are one long record each, of the force and of the mirror with $Q = 10$.
- Write a loop over $k = 0, 1, \dots, k_{\max}$ that fills two arrays, `r_F` and `r_x`. Each holds the time average of $x[n]\,x[n+k]$, for the force and for the mirror. Start from `np.zeros(k_max + 1)`.
- The slice `x[:N - k]` is $x[n]$ and `x[k:]` is $x[n+k]$. Each has $N - k$ samples, so `np.mean(x[:N - k] * x[k:])` is the average.

**Print.**
- $r_{xx}[0]$ of each.
- The fraction of the lags $k > 0$ of the force inside $\pm 2\sigma_{F}^{2}/\sqrt{N}$.
- $r_{xx}[k]/r_{xx}[0]$ of the mirror after half a period, $k\Delta t = 0.5$ s.

**What you should see.**
- The force has one spike at $k = 0$ of height $\sigma_{F}^{2} = 1600$.
- About 95 % of the other lags lie inside the band.
- The mirror has a variance of about $0.61$ a.u.$^{2}$.
- After half a period $r_{xx}[k]/r_{xx}[0]$ is near $-0.85$.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
# the autocorrelation as the time average of x[n] x[n+k], for k = 0 ... k_max: first the force, then the mirror
r_F = np.zeros(k_max + 1)
r_x = np.zeros(k_max + 1)
for k in range(k_max + 1):
    r_F[k] = np.mean(force[:N - k] * force[k:])
    r_x[k] = np.mean(x[:N - k] * x[k:])

# the force: r[0] is sigma_F^2 and every other lag is small; the mirror: the variance and half a swing later
band = 2 * sigma_F ** 2 / np.sqrt(N)
inside = np.mean(np.abs(r_F[1:]) < band)
print(f"force:  r[0] = {r_F[0]:.1f}   sigma_F^2 = {sigma_F ** 2:.1f}   fraction of the lags k > 0 inside +-{band:.1f}: {inside:.2f}")
print(f"mirror: r[0] = {r_x[0]:.4f} a.u.^2   r(0.5 s) / r[0] = {r_x[int(0.5 / dt)] / r_x[0]:.3f}")
```
**In one line.** The force has a spike at $k = 0$ and nothing else that matters. The mirror has $r_{xx}[0] = 0.6108$ a.u.$^{2}$ and a ratio of $-0.852$ at 0.5 s.

| quantity | value |
|---|---|
| $r_{xx}[0]$ of the force | $1596$, which is $\sigma_{F}^{2} = 1600$ |
| scatter of $r_{xx}[k]$ of the force for $k \ge 1$ | $\sigma_{F}^{2}/\sqrt{N} = 7.2$ |
| lags of the force inside twice that, $\pm 14.3$ | $96\,\%$ |
| $r_{xx}[0]$ of the mirror, its variance | $0.6108$ a.u.$^{2}$ |
| $r_{xx}[k]/r_{xx}[0]$ of the mirror at 0.5 s | $-0.852$ |

**Why.**
- For $k \ge 1$ the force is not exactly zero. Each $r_{xx}[k]$ is an average of about $N$ products of independent numbers.
- The exact autocorrelation of white noise is $\sigma_{F}^{2}\,\delta[k]$.
- Half a period later the mirror is on the other side.

</details>


In [ ]:
# the same autocorrelation with the fast function: it must agree with your loop
r_mirror = [autocorrelation(rec[1][0], k_max) for rec in records]
print("your loop equals the fast function:", np.allclose(r_x, r_mirror[1]) and np.allclose(r_F, autocorrelation(force, k_max)))
print("   b (kg/s)      Q   tau0 (s)   r[0] (a.u.^2)   r(0.5 s)/r[0]")
for bb, r in zip(b_values, r_mirror):
    f0, Q, tau0 = oscillator_parameters(m, k_spring, bb)
    print(f"  {bb:8.2f}  {Q:5.1f}  {tau0:9.2f}  {r[0]:14.4f}  {r[int(0.5 / dt)] / r[0]:13.3f}")
print("scatter plots of the mirror with b = %.2f:" % b)
for lag_s in lags_s:
    k = int(round(lag_s / dt))
    rho = np.corrcoef(x[:-k], x[k:])[0, 1]
    print(f"  lag {k * dt:5.2f} s: correlation {rho:6.3f}   envelope exp(-k dt / tau0) = {np.exp(-k * dt / (2 * m / b)):.3f}")

# top: x[n+k] against x[n] for three lags; bottom: the autocorrelation of the force and of the mirror, divided by r[0]
lag = np.arange(k_max + 1) * dt
colours = [P['BLUE'], P['BRICK'], P['SAGE']]
fig, axs = plt.subplot_mosaic([['s0', 's1', 's2'], ['acf', 'acf', 'acf']], figsize=(11, 8.5),
                              gridspec_kw={'height_ratios': [1, 1.1]})
half = 3.2 * x.std()
for name, lag_s in zip(('s0', 's1', 's2'), lags_s):
    k = int(round(lag_s / dt))
    rho = np.corrcoef(x[:-k], x[k:])[0, 1]
    axs[name].plot(x[:2500], x[k:k + 2500], '.', color=P['BLUE_MID'], ms=3, alpha=0.5)
    axs[name].set(xlim=(-half, half), ylim=(-half, half), aspect='equal', xlabel='$x[n]$ (a.u.)')
    axs[name].set_title(f'lag $k\\,\\Delta t$ = {k * dt:.2f} s, correlation {rho:.2f}', fontsize=10, loc='left')
axs['s0'].set_ylabel('$x[n+k]$ (a.u.)')
axs['acf'].plot(lag, r_F / r_F[0], color=P['GREY_MID'], lw=2.0, label='force (white)')
for bb, r, c in zip(b_values, r_mirror, colours):
    f0, Q, tau0 = oscillator_parameters(m, k_spring, bb)
    axs['acf'].plot(lag, r / r[0], color=c, lw=1.4, label=f'mirror, $Q$ = {Q:.0f}')
    axs['acf'].plot(lag, np.exp(-lag / tau0), color=c, lw=0.9, ls='--')
    axs['acf'].plot(lag, -np.exp(-lag / tau0), color=c, lw=0.9, ls='--')
axs['acf'].set(xlabel='lag $k\\,\\Delta t$ (s)', ylabel=r'$r_{xx}[k]\,/\,r_{xx}[0]$ (no unit)', xlim=(0, lag_max_s))
axs['acf'].legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=4, frameon=False)
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time. Run the cells of the task again each time.
1. Nothing first. Look at the three clouds, then at the autocorrelation of the force and of the mirror.
2. `lags_s`: try (0.02, 0.24, 3.0) s. Which cloud is round, a quarter of a period later? Why is the cloud at 3 s a line again?
3. `b` and `b_others`: halve each one. How does the memory of each curve change?
4. `k_spring` (try 10 N/m): how does the rhythm of the autocorrelation change?

**Question.** The force has no memory and the mirror has. What does the autocorrelation of each one look like? What sets the length of the memory of the mirror?

<details>
<summary>Solution</summary>

**In one line.** The force forgets at once: a spike at $k = 0$ (the gray line). The mirror remembers: its autocorrelation rings at $f_{0} = 1$ Hz and fades inside the envelope $\mathrm{e}^{-k\Delta t/\tau_{0}}$.

| | $Q = 5$ | $Q = 10$ | $Q = 20$ |
|---|---|---|---|
| $\tau_{0}$ (s) | $1.59$ | $3.17$ | $6.45$ |
| first minimum, after half a period, 0.5 s | $-0.73$ | $-0.85$ | $-0.92$ |
| variance $r_{xx}[0]$ | $0.310$ | $0.611$ | $1.229$ |

The variance grows with the memory. The fast function agrees with your loop: `True`.

**The three clouds.**

| lag | correlation | cloud |
|---|---|---|
| $0.04$ s | $0.97$ | a line |
| $0.5$ s | $-0.85$ | the line leans the other way |
| $10$ s | $0.04$ | round: 10 s is three times $\tau_{0}$ and the envelope is $0.043$ |

**With the lags $(0.02, 0.24, 3.0)$ s.** The correlations are $0.99$, $0.10$ and $0.37$.
- After a quarter of a period the cloud is round although the envelope is still $0.93$. The position is no longer correlated with the earlier position. It now follows the earlier velocity.
- At 3 s, three full periods later, the mirror is back in phase and the cloud is a line again. The envelope thins it: $\mathrm{e}^{-3/3.17} = 0.39$.

**Other changes.**
- If $b$ is halved the memory doubles: $\tau_{0} = 6.35$ s for the middle curve.
- With $k_{\mathrm{s}} = 10$ N/m, $f_{0} = 0.50$ Hz and the first minimum moves from 0.5 s to 1.0 s.

**What the autocorrelation says.**
- The force forgets at once.
- The mirror remembers for a time $\tau_{0} = 2Q/\omega_{0} = 2m/b$ and rings at $f_{0}$ while it remembers, as the ringing of H02.
- The damping and the mass set the memory. A high $Q$ remembers longer.

</details>


## Where the power is

The autocorrelation says how the samples are linked in time. The power spectral density says where the power is in frequency. We measure it from its definition.

### Task 3 · The spectrum

**Uses** `oscillator_parameters`, `transfer_function` and `kicked_mirror` from task 1.

**What we do.** The power spectral density is the expectation of the squared transform of a record divided by its length. The transform of $N$ samples taken every $\Delta t$ gives, at the frequencies $f_{j}$, the periodogram

$$
\hat{S}_{1}(f_{j}) = \frac{2\,\Delta t}{N}\left|\sum_{n=0}^{N-1} x[n]\,\mathrm{e}^{-\mathrm{i} 2\pi j n/N}\right|^{2}, \qquad f_{j} = \frac{j}{N\Delta t}, \quad j = 0, 1, \dots, N/2 .
$$

The factor $2\,\Delta t$ gives it in a.u.$^{2}$ per hertz, one sided.

**The steps.**
1. We have $R$ records. We follow the definition: we average $\hat{S}_{1}$ over the records.
2. For the white force the average is flat at $S_{F} = 2\,\sigma_{F}^{2}\,\Delta t$.
3. For the mirror, lecture 9 gives the density that comes out of a linear system:

$$
S_{xx}(f) = |G_{F}(f)|^{2}\,S_{F} .
$$

**The figure.** The averages for the force and for the mirror with three values of $Q$. The dashed lines are the formula.

**The area.** Lecture 9 also says that the area under the density is the variance, $r_{xx}[0]$. The cell prints the areas.

> **Careful — The two end points**
>
> The factor 2 in $\hat{S}_{1}$ is there because every frequency $f_{j}$ has a twin at $-f_{j}$ with the same power. The frequencies $f = 0$ and $f = f_{s}/2$ have no twin. So their values are not doubled: divide the first and the last value of $\hat{S}_{1}$ by 2. Do this every time you build a one sided density by hand. Without it the area is not exactly the variance. `scipy` does it for you.

> **The symbols**
>
> - **$\hat{S}_{1}(f_{j})$** — the periodogram of one record: one sided, per hertz; the hat means that it is estimated from data
> - **$f_{j}$**, **$j$** — the frequencies of the transform and their index; **$\delta f = 1/(N\Delta t)$** is the step between two of them, the resolution; $f_{s} = 1/\Delta t$ is the sampling frequency

The steps of this task.

![R records, the squared transform of each, the average over the records, compared with the formula.](figures/h03_task3_route.png)

In [ ]:
def output_psd(f, m, k_spring, b, S_F):
    """Return the density of the mirror from the density of the force: S_xx = |G_F|^2 S_F.

    Uses transfer_function from task 1.

    Parameters
    ----------
    f        : array, frequency (Hz)
    m, k_spring, b : mass (kg), spring constant (N/m), damping coefficient (kg/s)
    S_F      : float or array, the density of the force (a.u.^2 per hertz)

    Returns
    -------
    S_xx : array like f, the density of the mirror (a.u.^2 per hertz)
    """
    # the power gain is the square of the amplitude gain
    S_xx = np.abs(transfer_function(f, m, k_spring, b)) ** 2 * S_F
    return S_xx

In [ ]:
# change these: the mirror, with three dampings, and the records
m = 1.0                          # mass of the mirror (kg)
k_spring = 39.5                  # spring constant (N/m): f0 = 1 Hz
b_values = (1.26, 0.63, 0.31)    # damping coefficients (kg/s): Q = 5, 10, 20
sigma_F = 40.0                   # rms of the force in one sample (a.u.)
dt = 0.02                        # time step (s)
R = 200                          # number of records
N = 8192                         # samples in one record: 164 s

# the same R forces push the three mirrors; sets holds the force and the three positions, each of shape (R, N)
sets = [kicked_mirror(R, N, dt, m, k_spring, b_values[0], sigma_F)[0]]
sets += [kicked_mirror(R, N, dt, m, k_spring, bb, sigma_F)[1] for bb in b_values]
f = np.fft.rfftfreq(N, dt)
delta_f = f[1] - f[0]
print(f"{len(f)} frequencies, {delta_f:.5f} Hz apart, up to {f[-1]:.0f} Hz")

**Your code.** `sets` is a list of four arrays of shape $(R, N)$: the forces and the positions of the mirror for the three dampings. Write a loop over `sets`. For each array:
1. Take `np.fft.rfft(..., axis=-1)`.
2. Scale its square as in $\hat{S}_{1}$.
3. Divide the first and the last value by 2 (`S[:, 0]` and `S[:, -1]`).
4. Average over the records with `S.mean(axis=0)` and append the result to the list `S_measured`.

**Print.**
- The mean of the density of the force without its two end points.
- The area of the mirror with $Q = 10$, `np.sum(S) * delta_f`, next to the mean square of its records, `np.mean(sets[2] ** 2)`.

**What you should see.**
- The density of the force is flat at $2\sigma_{F}^{2}\Delta t = 64$ a.u.$^{2}$/Hz.
- The area of the mirror is its mean square: the two numbers are equal to every printed digit.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
# for each set: the squared transform of every record, scaled to one sided per hertz, then the average over the records
S_measured = []
for records in sets:
    X = np.fft.rfft(records, axis=-1)
    S = 2 * dt * np.abs(X) ** 2 / N
    # f = 0 and f = fs/2 have no twin at -f: they are not doubled
    S[:, 0] = S[:, 0] / 2
    S[:, -1] = S[:, -1] / 2
    S_measured.append(S.mean(axis=0))

# the level of the force without its two end points; the area of the mirror with Q = 10 against its mean square
S_F_measured = S_measured[0][1:-1].mean()
area = np.sum(S_measured[2]) * delta_f
print(f"force: mean of the density {S_F_measured:.2f}   2 sigma_F^2 dt = {2 * sigma_F ** 2 * dt:.2f} a.u.^2/Hz")
print(f"mirror, Q = 10: area under the density {area:.6f}   mean square of the records {np.mean(sets[2] ** 2):.6f} a.u.^2")
```
**In one line.** The area under the density of the mirror is its mean square, exactly.

| quantity | value |
|---|---|
| mean of the density of the force | $64.00$, which is $2\sigma_{F}^{2}\Delta t = 2 \times 1600 \times 0.02$ |
| area of the mirror | $0.656096$ |
| mean square of its records | $0.656096$ a.u.$^{2}$ |

**Why.**
- The sum over $j$ of the squared transform is $N$ times the sum of the squared samples.
- So the equality is exact for every record when the two end points are halved.
- It is the discrete form of "the area under the density is $r_{xx}[0]$".

</details>


In [ ]:
# the density of the force and of the mirror for three Q on the same axes; dashed: the formula |G_F|^2 S_F
S_F = 2 * sigma_F ** 2 * dt
S_formula = [output_psd(f, m, k_spring, bb, S_F) for bb in b_values]
colours = [P['BLUE'], P['BRICK'], P['SAGE']]
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.loglog(f[1:], S_measured[0][1:], color=P['GREY_MID'], lw=1.6, label='force (white)')
for bb, S_m, S_f, c in zip(b_values, S_measured[1:], S_formula, colours):
    f0, Q, tau0 = oscillator_parameters(m, k_spring, bb)
    ax.loglog(f[1:], S_m[1:], color=c, lw=1.2, label=f'mirror, $Q$ = {Q:.0f}')
    ax.loglog(f[1:], S_f[1:], color=P['INK'], lw=0.9, ls='--')
handles = ax.get_legend_handles_labels()[0]
handles += [Line2D([], [], color=P['INK'], lw=0.9, ls='--', label=r'$|G_F|^2\,S_F$')]
ax.legend(handles=handles, loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=5, frameon=False, fontsize=9)
ax.axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
ax.set(xlabel='frequency (Hz)', ylabel=r'$S_{xx}(f)$ (a.u.$^2$/Hz)', xlim=(0.05, 25), ylim=(1e-8, 1e2))
plt.tight_layout()
plt.show()

# the numbers: the peak of each mirror against the formula, the scatter around the formula and the areas
print("   Q    S at f0 measured   |G_F|^2 S_F = (Q / k_s)^2 S_F   f0/Q (Hz)   area measured   area formula")
for bb, S_m, S_f in zip(b_values, S_measured[1:], S_formula):
    f0, Q, tau0 = oscillator_parameters(m, k_spring, bb)
    i0 = np.argmin(np.abs(f - f0))
    print(f"{Q:5.1f}  {S_m[i0]:16.3f}  {S_f[i0]:28.3f}  {f0 / Q:10.3f}  {np.sum(S_m) * delta_f:13.4f}  {np.trapezoid(S_f, f):12.4f}")
band = (f > 0.3) & (f < 10)
print(f"scatter of the measured curve around the formula, 0.3 to 10 Hz: {np.std(S_measured[2][band] / S_formula[1][band]):.3f}   1 / sqrt(R) = {1 / np.sqrt(R):.3f}")
print(f"at low frequency: S_xx = S_F / k_s^2 = {S_F / k_spring ** 2:.4f}   measured, mean below 0.2 Hz: {S_measured[2][(f > 0) & (f < 0.2)].mean():.4f}")

**What to change.** On the line `# change these`, one parameter at a time. Run the cells of the task again each time.
1. Nothing first. Read the level of the force in the printed line. Then compare the three mirrors at $f_{0}$ and far above it.
2. `R`: 20 and 2000. What happens to the scatter of the measured curves around the dashed ones?
3. `sigma_F`: 80. Which curves move and by how much?
4. `b_values`: try (3.15, 0.63, 0.063). How do the height and the width of the peak follow $Q$?

**Question.** Why is the density of the force flat and the density of the mirror peaked? What sets the height and the width of the peak?

<details>
<summary>Solution</summary>

**In one line.** The density of the force is flat. The density of the mirror has a peak at $f_{0}$ of height $(Q/k_{\mathrm{s}})^{2}S_{F}$ and width $f_{0}/Q$.

| | $Q = 5.0$ | $Q = 10.0$ | $Q = 20.3$ |
|---|---|---|---|
| measured density at $f_{0}$ | $1.07$ | $4.27$ | $17.6$ |
| formula $(Q/k_{\mathrm{s}})^{2}S_{F}$ | $1.02$ | $4.08$ | $16.8$ |
| width at half height $f_{0}/Q$ (Hz) | $0.20$ | $0.10$ | $0.049$ |
| area under the measured curve | $0.326$ | $0.656$ | $1.347$ |
| area under the formula | $0.322$ | $0.643$ | $1.307$ |

The areas grow with $Q$.

**With $b = (3.15, 0.63, 0.063)$.**

| | $Q = 2.0$ | $Q = 10.0$ | $Q = 99.8$ |
|---|---|---|---|
| peak | $0.171$ | $4.27$ | $419$ |
| formula | $0.163$ | $4.08$ | $408$ |

**Away from the peak.**
- Far above $f_{0}$ the three curves fall together as $f^{-4}$.
- Far below $f_{0}$ the density is $S_{F}/k_{\mathrm{s}}^{2} = 0.0410$. The measured mean below 0.2 Hz is $0.0421$: the mirror follows the force like a spring.

**The scatter.**
- The measured curves scatter around the formula by $0.071$, which is $1/\sqrt{R}$.
- With $R = 20$ it is $0.221$ and with $R = 2000$ it is $0.022$.

**With $\sigma_{F} = 80$.** The force moves to $256$ and every curve moves up by four. The level scales as $\sigma_{F}^{2}$ and the shape does not change.

**Why flat and why peaked.**
- The kicks are independent, so all frequencies have the same power: the density of the force is flat.
- The mirror answers each frequency with the gain $|G_{F}|$, so the power is multiplied by $|G_{F}|^{2}$.
- Below $f_{0}$ the gain is $1/k_{\mathrm{s}}$ and the mirror follows the force.
- At $f_{0}$ the gain is $Q/k_{\mathrm{s}}$ and the power is $Q^{2}$ times larger.
- Above $\sqrt{2}f_{0}$ the mirror is quieter than the low frequency level.
- The density falls to half of its peak where $|G_{F}|$ falls by $\sqrt{2}$. This is the half-power width $f_{0}/Q$ of H02.
- A higher $Q$ gives a higher and narrower peak.
- The memory $\tau_{0}$ of task 2 and the width $f_{0}/Q$ are one fact seen in time and in frequency.

</details>


## The estimate from one record

A detector gives one record. We cut it into segments and we average the periodograms of the segments. We see how long the segments must be.

### Task 4 · One record: Welch and the choice of $D$

**Uses** `oscillator_parameters`, `kicked_mirror` from task 1 and `output_psd` from task 3.

**What we do.** In task 3 we averaged over many records. A detector gives one record of $N$ samples. Its periodogram has two errors.
- The **bias** is the mean of the estimate minus the true $S_{xx}(f)$.
- The **variance** is the spread of the estimate around its mean.

The periodogram of the whole record has a standard deviation as large as its mean. A longer record does not lower it.

**Welch's method.** It lowers the variance.
1. Cut the record into $K$ segments of $D$ samples. They overlap by half.
2. Multiply each segment by a window $H[n]$ and take its periodogram.
3. Average the $K$ results.
4. Divide the estimate by the mean square of the window, so the level stays right.

**The window.** Here it is Hann. It goes smoothly to zero at both ends and it stops a loud line from leaking over the band.

**The trade.**
- The variance falls as $1/K$.
- A segment of $D$ samples smooths the spectrum over the resolution $\delta f = 1/(D\,\Delta t)$. No detail finer than that remains. This is the bias.
- The peak of the mirror is $f_{0}/Q$ wide.

**In the code.**
- `signal.welch` does all of this. It does not double $f = 0$ and $f_{s}/2$, as in task 3.
- By default it subtracts the mean of every segment first. We keep the mean with `detrend=False`: our process has zero mean. Removing the mean of a short segment would also remove part of the slow motion.
- We cut one long record with three values of $D$.

**The figure.** The periodogram of the whole record, the three Welch estimates and the formula on the same axes. On the right we zoom on the resonance.

> **The symbols**
>
> - **$D$** — the number of samples in one segment; **$K$** — the number of segments
> - **$H[n]$** — the window: $\sin^{2}(\pi n/D)$ for Hann
> - **$\hat{S}_{W}(f)$** — the Welch estimate: the average of the windowed periodograms of the $K$ segments
> - **$\delta f = 1/(D\,\Delta t)$** — the frequency step of a segment, the resolution

The steps of this task.

![The record cut into K segments of D samples, each windowed and squared, then averaged.](figures/h03_task4_route.png)

In [ ]:
def welch_numbers(f, S, D, overlap, N, dt, m, k_spring, b, S_F):
    """Return the numbers that describe one Welch estimate of the mirror.

    Uses output_psd from task 3.

    Parameters
    ----------
    f, S     : arrays, the frequencies (Hz) and the estimate (a.u.^2 per hertz) from signal.welch
    D, N     : int, samples in one segment and in the record
    overlap  : float, the fraction of a segment shared with the next one
    dt       : float, time step (s)
    m, k_spring, b : mass (kg), spring constant (N/m), damping coefficient (kg/s) of the mirror
    S_F      : float, the density of the force (a.u.^2 per hertz)

    Returns
    -------
    (step, K, scatter, peak, area) : step the frequency step 1 / (D dt) (Hz); K the number of segments;
        scatter the standard deviation of S / formula - 1 between 10 and 20 Hz, where the density is smooth;
        peak the largest S near f0 divided by the true height at f0; area the sum of S times the step (a.u.^2)
    """
    # the step and the number of segments
    step = f[1] - f[0]
    K = (N - D) // int(D * (1 - overlap)) + 1
    # the scatter in a smooth band, where only the variance matters
    smooth = (f > 10) & (f < 20)
    scatter = np.std(S[smooth] / output_psd(f[smooth], m, k_spring, b, S_F) - 1)
    # the height at the peak against the true one, where the bias shows
    f0 = oscillator_parameters(m, k_spring, b)[0]
    i0 = np.argmin(np.abs(f - f0))
    peak = S[i0 - 1:i0 + 2].max() / output_psd(f0, m, k_spring, b, S_F)
    # the area under the estimate: its end points are already right
    area = np.sum(S) * step
    numbers = (step, K, scatter, peak, area)
    return numbers

In [ ]:
# change these: the mirror, the record and the segments
m = 1.0                    # mass of the mirror (kg)
k_spring = 39.5            # spring constant (N/m): f0 = 1 Hz
b = 0.63                   # damping coefficient (kg/s): Q = 10, so the peak is 0.1 Hz wide
sigma_F = 40.0             # rms of the force in one sample (a.u.)
dt = 0.02                  # time step (s)
N = 2 ** 17                # samples in the record: 44 minutes
D_values = (64, 512, 4096) # samples in one segment: the frequency step is 1 / (D dt) = 0.78, 0.098, 0.012 Hz
overlap = 0.5              # fraction of a segment shared with the next one

# one long record of the mirror and the periodogram of the whole of it
f0, Q, tau0 = oscillator_parameters(m, k_spring, b)
fs = 1 / dt
x = kicked_mirror(1, N, dt, m, k_spring, b, sigma_F)[1][0]
f_all, S_all = signal.periodogram(x, fs=fs, window='boxcar', detrend=False)
S_F = 2 * sigma_F ** 2 * dt
print(f"f0 = {f0:.3f} Hz   Q = {Q:.1f}   f0/Q = {f0 / Q:.3f} Hz   N = {N} samples   variance of the record {x.var():.4f} a.u.^2")

**Your code.**
- `D_values` holds three values of $D$. Write a loop over them.
- In each turn call `signal.welch(x, fs=fs, window='hann', nperseg=D, noverlap=int(D * overlap), detrend=False)`. It returns the frequencies and the estimate.
- Append that pair to the list `estimates`.
- The next cell prints a table and draws the figure from `estimates`.

**What you should see.**
- The periodogram of the whole record scatters by 100 %.
- A short segment gives a smooth curve and a step of 0.78 Hz, eight times wider than the peak: the peak is lost.
- A long segment gives a step of 0.012 Hz and the right peak but more scatter. The scatter is close to $1/\sqrt{K}$.
- The area under every estimate is close to the variance of the record.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
# Welch with Hann for every D: the estimate of each is a pair (f, S)
estimates = []
for D in D_values:
    f_w, S_w = signal.welch(x, fs=fs, window='hann', nperseg=D, noverlap=int(D * overlap), detrend=False)
    estimates.append((f_w, S_w))
```

</details>


In [ ]:
# the table: the step, K, the scatter against 1 / sqrt(K), the height of the peak and the area against the variance
numbers = [welch_numbers(f_w, S_w, D, overlap, N, dt, m, k_spring, b, S_F) for (f_w, S_w), D in zip(estimates, D_values)]
smooth = (f_all > 10) & (f_all < 20)
scatter_all = np.std(S_all[smooth] / output_psd(f_all[smooth], m, k_spring, b, S_F) - 1)
print(f"whole record, one periodogram: scatter {scatter_all:.2f}   variance of the record {x.var():.4f}")
print("     D     K   step 1/(D dt) (Hz)   scatter   1/sqrt(K)   peak / true   area (a.u.^2)")
for D, (step, K, scatter, peak, area) in zip(D_values, numbers):
    print(f"{D:6d}  {K:5d}  {step:18.4f}  {scatter:8.3f}  {1 / np.sqrt(K):10.3f}  {peak:11.2f}  {area:13.4f}")

# left: the periodogram, the Welch estimates and the formula on the same axes; right: the resonance enlarged
colours = [P['BLUE'], P['SAGE'], P['BRICK']]
f_f = np.geomspace(0.02, 25, 3000)
formula = output_psd(f_f, m, k_spring, b, S_F)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 4.8))
axa.loglog(f_all[1:], S_all[1:], color=P['GREY_LIGHT'], lw=0.8, label='periodogram of the whole record')
near_f0 = np.abs(f_f - f0) < 0.6
for D, (f_w, S_w), c in zip(D_values, estimates, colours):
    axa.loglog(f_w[1:], S_w[1:], color=c, lw=1.5, label=f'Welch, $D$ = {D}')
    axb.plot(f_w[1:], S_w[1:], 'o-', color=c, lw=1.5, ms=3.5, label=f'$D$ = {D}')
axa.plot(f_f, formula, color=P['INK'], lw=1.0, ls='--', label=r'$|G_F|^2 S_F$')
axb.plot(f_f[near_f0], formula[near_f0], color=P['INK'], lw=1.0, ls='--', label=r'$|G_F|^2 S_F$')
axa.set(xlabel='frequency (Hz)', ylabel=r'$\hat{S}(f)$ (a.u.$^2$/Hz)', xlim=(0.05, 25), ylim=(1e-8, 1e2))
axa.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False, fontsize=9)
axb.set(xlabel='frequency (Hz)', ylabel=r'$\hat{S}(f)$ (a.u.$^2$/Hz)', xlim=(f0 - 0.6, f0 + 0.6), ylim=(0, 1.3 * output_psd(f0, m, k_spring, b, S_F)))
axb.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=4, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

<details>
<summary>Solution</summary>

**In one line.** A short segment gives a small scatter and loses the peak. A long segment gives the right peak and a larger scatter.

**The call.**
- `signal.welch` returns the pair $(f, \hat{S}_{W})$.
- `window='hann'` is the Hann window.
- `noverlap` is the number of samples shared by two neighboring segments.
- `fs=fs` gives the frequencies in hertz and the density per hertz, one sided.

| $D$ | step $\delta f$ | $K$ | scatter | $1/\sqrt{K}$ | peak / true | area |
|---|---|---|---|---|---|---|
| 64 | 0.781 Hz | 4095 | 0.016 | 0.016 | 0.12 | 0.6551 |
| 512 | 0.098 Hz | 511 | 0.049 | 0.044 | 0.62 | 0.6533 |
| 4096 | 0.012 Hz | 63 | 0.129 | 0.126 | 1.09 | 0.6535 |

The scatter is measured between 10 and 20 Hz, where the density is smooth. The periodogram of the whole record scatters by $1.00$.

**Why.**
- The variance of the record is $0.6540$ a.u.$^{2}$. Every estimate has an area very close to it: the bias moves power sideways and does not remove it.
- With $D = 64$ the density is smoothed over eight times the width of the peak. The peak is only $12\,\%$ of its true height and the power it lost is in the nearby bins.
- With $D = 512$ the step is the width of the peak and the peak is $38\,\%$ too low.
- With $D = 4096$ the step is a tenth of the width and the peak is right within the scatter.

</details>


**Your choice.** Read the table. Choose a value of $D$ and call it `D_best`.
- The step must be much smaller than the width $f_{0}/Q$ of the peak.
- The scatter must stay below about 15 %.

Call `signal.welch` once more with `D_best`. Print the step as a fraction of $f_{0}/Q$, $K$, the scatter, the height of the peak and the area against the variance. The function `welch_numbers` gives them: it returns `(step, K, scatter, peak, area)`.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
# the choice: a step ten times finer than the peak and about 60 segments
D_best = 4096
f_b, S_b = signal.welch(x, fs=fs, window='hann', nperseg=D_best, noverlap=D_best // 2, detrend=False)
step, K, scatter, peak, area = welch_numbers(f_b, S_b, D_best, 0.5, N, dt, m, k_spring, b, S_F)
print(f"D = {D_best}: step {step:.4f} Hz = {step / (f0 / Q):.2f} of the width f0/Q   K = {K}   scatter {scatter:.3f}   peak / true {peak:.2f}")
print(f"area under the estimate {area:.4f}   variance of the record {x.var():.4f} a.u.^2")
```
**In one line.** $D = 4096$ resolves the peak and keeps the scatter below 15 %.

| quantity | value |
|---|---|
| step | $0.0122$ Hz, which is $0.12$ of the width $f_{0}/Q = 0.100$ Hz |
| segments $K$ | $63$ |
| scatter | $0.129$ |
| peak | $1.09$ of the true one: right within the scatter |
| area against the variance | $0.6535$ against $0.6540$ |

**Why.**
- $D = 512$ fails the step and $D = 64$ fails it badly.
- A larger $D$ than 4096 would lower the bias further and raise the scatter.

</details>


**What to change.** On the line `# change these`, one parameter at a time. Run the cells of the task again each time.
1. Nothing first. Read the table: the scatter against $1/\sqrt{K}$ and the height of the peak against the true one. On the right plot which curve has the right peak?
2. `D_values`: try (32, 128, 1024). How do the scatter and the peak move?
3. `overlap`: 0 and 0.75. What changes in the scatter?
4. `N`: $2^{19}$. Which of the two errors can a longer record lower?
5. `b` (try 0.126 kg/s, so $Q = 50$) with `D_values = (512, 4096, 16384)`: the peak is 0.02 Hz wide. Which $D$ resolve it?

**Question.** Why does a short segment give a small scatter and a wrong peak while a long segment gives the right peak and a large scatter? How do we choose $D$?

<details>
<summary>Solution</summary>

**In one line.** Segments that are short give many segments and a small scatter but a wrong peak. Long segments give the right peak and a large scatter. The narrowest feature sets $D$.

**With $D = (32, 128, 1024)$.**

| $D$ | 32 | 128 | 1024 |
|---|---|---|---|
| peak / true | $0.06$ | $0.18$ | $0.79$ |
| scatter | $0.059$ | $0.019$ | $0.067$ |

**With the overlap changed** ($D = 64$, $512$, $4096$).

| overlap | $K$ | scatter | $1/\sqrt{K}$ |
|---|---|---|---|
| none | $2048$, $256$, $32$ | $0.025$, $0.061$, $0.169$ | $0.022$, $0.063$, $0.177$ |
| $0.75$ | $8189$, $1021$, $125$ | $0.012$, $0.045$, $0.123$ | |

- Without overlap the scatter is close to $1/\sqrt{K}$.
- With overlap $0.75$ the scatter is above $1/\sqrt{K}$ for the last two: segments that share most of their samples are not independent.
- Half overlap is a good compromise.

**With $N = 2^{19}$.**
- The scatter falls to $0.008$, $0.022$ and $0.060$.
- The peak stays at $0.12$, $0.62$ and $1.07$ of the true one.
- A longer record lowers the variance. It does not move the bias at a fixed $D$.

**With $b = 0.126$ kg/s** and $D = 512$, $4096$ and $16384$.
- The peak is $0.18$, $0.74$ and $1.26$ of the true height.
- Only $D = 16384$, with a step of $0.003$ Hz, resolves it. The cost is $K = 15$ segments and a scatter of $0.26$.

**How we choose $D$.**
- A short segment gives many segments, so a small variance. But it smooths the density over $\delta f = 1/(D\Delta t)$ and loses the peak: a bias.
- A long segment keeps the peak and gives few segments, so a large variance.
- For a fixed record we cannot have both small.
- The narrowest feature we must resolve, here $f_{0}/Q$, sets $D$: the step must be much smaller than the width.
- Then $K \approx N/D \times 2$ gives the scatter $1/\sqrt{K}$.
- Only a longer record lowers both errors.

</details>


## In class, on your own

### Exercise B · The same noise in a circuit

**The noise.**
- A resistor is at a temperature: the charges in it move at random.
- This gives a random voltage $v$ across the resistor, the thermal noise voltage.
- The kicks are independent, so $v$ is white noise.

**The circuit.** We put the resistor in series with an inductor and a capacitor. The noise voltage $v$ drives the circuit. The charge $q$ on the capacitor obeys

$$
L\,\ddot{q} + R_{\mathrm{res}}\,\dot{q} + \frac{q}{C} = v .
$$

It is the equation of the mirror with other names. There is no code to read here: you use the functions that you already have.

> **The symbols**
>
> - **$L$**, **$C$**, **$R_{\mathrm{res}}$** — the inductance (H), the capacitance (F) and the resistance (ohm) of the circuit; $R_{\mathrm{res}}$ is not the number of records
> - **$v$**, **$q$** — the white noise voltage in a.u. and the charge on the capacitor; the voltage across the capacitor is $v_{C} = q/C$
> - **$\sigma_{v}$** — the rms of the voltage in one sample, in a.u.

![A series circuit: a noise voltage, a resistor, an inductor and a capacitor. We record the voltage across the capacitor.](figures/h03_circuit_route.png)

![The steps of the exercise: the white noise voltage drives the circuit, one record of the voltage across the capacitor, the Welch estimate gives the resonance and the quality factor.](figures/h03_exercise_route.png)

**The data.**
- $L = 10$ mH, $C = 1$ $\mu$F and $R_{\mathrm{res}} = 10$ ohm.
- The noise has $\sigma_{v} = 1$ a.u. per sample and $\Delta t = 50$ $\mu$s.
- One record of $N = 2^{18}$ samples.

**What you do.**
1. Say which quantity of the circuit plays the role of $m$, $b$, $k_{\mathrm{s}}$, $F$ and $x$.
2. Compute $f_{0} = 1/(2\pi\sqrt{LC})$ and $Q = \sqrt{L/C}/R_{\mathrm{res}}$ from the formulas. Check them with `oscillator_parameters` and the roles of step 1.
3. Make the record of $v_{C} = q/C$ with `kicked_mirror`.
4. Estimate the density of $v_{C}$ with Welch and the Hann window. Choose $D$ so that the step is much smaller than the width of the peak.
5. Find $f_{0}$ from the position of the peak.
6. Find $Q$ in two ways: from the width of the peak at half height (it is $f_{0}/Q$) and from the height of the peak over the density at low frequency (it is $Q^{2}$). Compare with the formulas.
7. Check that the area under the estimate is the variance of $v_{C}$.
8. Draw the estimate with the formula $|G_{F}/C|^{2}S_{v}$ for the same record, where $S_{v} = 2\sigma_{v}^{2}\Delta t$.

**What you should see.**
- $f_{0} = 1591.5$ Hz and $Q = 10.0$.
- A peak at about 1.6 kHz and about 160 Hz wide, so $D = 1024$ (a step of 19.5 Hz) is enough.
- $Q$ from the width and from the height are both about 10.
- The density at low frequency is $S_{v} = 10^{-4}$ a.u.$^{2}$/Hz and the peak is about $10^{-2}$.
- The area and the variance of $v_{C}$ are both about $2.5$ a.u.$^{2}$.

In [ ]:
# your code here

<details>
<summary>Solution</summary>

```python
# the roles: L plays the mass m, R_res the damping b, 1/C the spring constant k_s, the voltage v the force F, the charge q the position x
L = 10e-3            # inductance (H)
C = 1e-6             # capacitance (F)
R_res = 10.0         # resistance (ohm)
sigma_v = 1.0        # rms of the voltage noise in one sample (a.u.)
dt = 5e-5            # time step (s): 20 kHz
N = 2 ** 18          # samples in the record: 13 s

# f0, Q and tau0 from the formulas, with the roles above
f0_formula = 1 / (2 * np.pi * np.sqrt(L * C))
Q_formula = np.sqrt(L / C) / R_res
f0_roles, Q_roles, tau0_roles = oscillator_parameters(L, 1 / C, R_res)
print(f"f0 = {f0_formula:.1f} Hz ({f0_roles:.1f} from the roles)   Q = {Q_formula:.2f} ({Q_roles:.2f})   tau0 = {1e3 * tau0_roles:.1f} ms")

# one record of the charge, then the voltage across the capacitor v_C = q / C
v_noise, q = kicked_mirror(1, N, dt, L, 1 / C, R_res, sigma_v)
v_C = q[0] / C
fs = 1 / dt

# Welch with Hann: D = 1024 gives a step of 19.5 Hz, eight steps across the peak, and K = 511 segments
D = 1024
f, S = signal.welch(v_C, fs=fs, window='hann', nperseg=D, noverlap=D // 2, detrend=False)
delta_f = f[1] - f[0]

# f0 from the position of the peak; Q from its width at half height and from its height over the low frequency level
peak = np.argmax(S)
f0_measured = f[peak]
width = np.sum(S > S[peak] / 2) * delta_f
Q_width = f0_measured / width
plateau = S[(f > 0) & (f < 0.2 * f0_formula)].mean()
Q_height = np.sqrt(S[peak] / plateau)
print(f"f0 from the peak {f0_measured:.0f} Hz   width at half height {width:.0f} Hz   f0/Q = {f0_formula / Q_formula:.0f} Hz")
print(f"Q from the width {Q_width:.1f}   Q from the height {Q_height:.1f}   formula {Q_formula:.1f}")

# the area under the estimate against the variance of the record, and the formula for the whole curve
S_v = 2 * sigma_v ** 2 * dt
formula = np.abs(transfer_function(f, L, 1 / C, R_res) / C) ** 2 * S_v
print(f"area {np.sum(S) * delta_f:.4f}   variance of v_C {v_C.var():.4f} a.u.^2   area of the formula {np.trapezoid(formula, f):.4f}")

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.semilogy(f[1:], S[1:], color=P['BLUE'], lw=1.4, label='Welch, Hann, $D$ = 1024')
ax.semilogy(f[1:], formula[1:], color=P['INK'], lw=1.0, ls='--', label=r'$|G_F / C|^2 S_v$')
ax.axvline(f0_formula, color=P['GREY_MID'], lw=0.8, ls=':')
ax.set(xlabel='frequency (Hz)', ylabel=r'$S_{xx}(f)$ of $v_C$ (a.u.$^2$/Hz)', xlim=(0, 6000), ylim=(1e-9, 1e-1))
ax.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False)
plt.show()
```
**In one line.** $L$ plays the mass, $R_{\mathrm{res}}$ the damping and $1/C$ the spring constant. The circuit has $f_{0} = 1591.5$ Hz and $Q = 10.00$.

**The roles.**

| circuit | mirror |
|---|---|
| $L$ | mass $m$ |
| $R_{\mathrm{res}}$ | damping $b$ |
| $1/C$ | spring constant $k_{\mathrm{s}}$ |
| voltage $v$ | force $F$ |
| charge $q$ | position $x$ |

- The call is `kicked_mirror(1, N, dt, L, 1 / C, R_res, sigma_v)`. It returns the voltage and the charge.
- The voltage across the capacitor is $q/C$ and its transfer function from $v$ is $G_{F}/C$, with a gain of 1 at low frequency.

**The numbers.**

| quantity | value |
|---|---|
| $f_{0}$ | $1591.5$ Hz |
| $Q$ | $10.00$ |
| $\tau_{0} = 2L/R_{\mathrm{res}}$ | $2.0$ ms |
| samples per period of $f_{0}$ (step of 50 $\mu$s) | $12.6$ |
| step of $D = 1024$ | $19.5$ Hz: eight steps across the peak |
| position of the peak | $1582$ Hz, within one step of $f_{0}$ |
| width at half height | $156$ Hz against $f_{0}/Q = 159$ Hz: $Q = 10.1$ |
| $Q$ from $\sqrt{S_{\mathrm{peak}}/S_{\mathrm{low}}}$ | $9.9$ |
| area under the estimate | $2.486$ a.u.$^{2}$ |
| variance of $v_{C}$ | $2.481$ |
| area under the formula | $2.500$ |

The values from the formulas and from the roles are the same.

**Why.** The voltage noise is $S_{v} = 10^{-4}$ a.u.$^{2}$/Hz. It is the same level as the low frequency plateau of $v_{C}$, because the gain is 1 there.

</details>


## What to hand in

**For each task.** The answer to the question, in two or three sentences. Add the parameters you changed and what you saw.

1. **Task 1.** Which average narrows with $R$ and which with $T$. When one record gives the expectation.
2. **Task 2.** What the autocorrelation of the force and of the mirror looks like. What sets the memory of the mirror.
3. **Task 3.** Why the density of the force is flat and the density of the mirror is peaked. What sets the height and the width of the peak.
4. **Task 4.** How the length of a segment balances bias against variance. How we choose it.
5. **Exercise B.** Which quantity plays which role. The values of $f_{0}$ and $Q$ that you measured and how close they are to the formulas.

> **Where this goes next**
>
> - The noise of the mirror is white noise through a transfer function.
> - Its autocorrelation and its density are one description in time and in frequency.
> - One record gives an estimate of the density with a bias and a variance. The segment length balances one against the other.
> - The noise of a detector is described and measured in the same way.

## Software

`numpy` throughout; `np.fft.rfft` gives the transforms of the tasks and `scipy.signal.welch` the estimates of the last one. The random processes follow (Therrien 1992) and (Papoulis and Pillai 2002) and the spectral estimates (Kay 1988). The window is described in (Harris 1978).

**References**

- Therrien (1992). *Discrete Random Signals and Statistical Signal Processing*.
- Papoulis and Pillai (2002). *Probability, Random Variables and Stochastic Processes*.
- Kay (1988). *Spectral Estimation*.
- Harris (1978). *On the use of windows for harmonic analysis with the discrete Fourier transform*.